In [1]:
import onnxruntime as ort
import json
import numpy as np


In [14]:
model_path = "nepali_grammar_checker_research.onnx"
tokenizer_path = "nepali_tokenizer_vocab_research.json"

session = ort.InferenceSession(model_path)

with open(tokenizer_path, "r", encoding="utf-8") as f:
    tokenizer = json.load(f)

word2idx = tokenizer["word2idx"]
idx2word = {int(k): v for k, v in tokenizer["idx2word"].items()}

def preprocess_input(text):
    tokens = text.split()
    if len(tokens) == 1:
        token_ids = [word2idx.get(text, word2idx["<UNK>"])]
    else:
        token_ids = [word2idx.get(tok, word2idx["<UNK>"]) for tok in tokens]
    return np.array(token_ids, dtype=np.int64).reshape(1, -1)

def postprocess_output(output):
    output = np.asarray(output)
    if output.ndim == 0:
        return str(output.item())
    token_ids = output.astype(int).flatten().tolist()
    return "".join(idx2word.get(tid, "<UNK>") for tid in token_ids if tid != word2idx["<PAD>"])

input_text = "तपाईं"
input_ids = preprocess_input(input_text)

input_meta = session.get_inputs()[0]
input_name = input_meta.name
seq_len = input_meta.shape[1]
if seq_len is None:
    seq_len = 20

if input_ids.shape[1] < seq_len:
    input_ids = np.pad(
        input_ids,
        ((0, 0), (0, seq_len - input_ids.shape[1])),
        constant_values=word2idx["<PAD>"],
    )
else:
    input_ids = input_ids[:, :seq_len]

outputs = session.run(None, {input_name: input_ids})
print(outputs)
corrected_text = postprocess_output(outputs[0])
print("Corrected Text:", corrected_text)
outputs[0][0]


[array([0.9999776], dtype=float32)]
Corrected Text: 


np.float32(0.9999776)